In [3]:
!pip install catboost sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.3/158.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.4/193.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/

In [4]:
import pandas as pd
import numpy as np
import torch
import random
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sdv.single_table import CTGANSynthesizer as CTGAN
from sdv.metadata import SingleTableMetadata
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

In [5]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

In [6]:
dir_datasets = 'data/processed/'

# Загрузка реальных датасетов
real_data_1 = pd.read_csv(dir_datasets+'abolone.csv')
real_data_2 = pd.read_csv(dir_datasets+'insurance.csv')
real_data_3 = pd.read_csv(dir_datasets+'two_moons.csv')

# Словарь датасетов для удобства
datasets = {
    'abolone': real_data_1,
    'insurance': real_data_2,
    #'two_moons': real_data_3
}

for name, data in datasets.items():
    print(f"{name}: {data.shape}, колонки: {list(data.columns)}")

abolone: (4177, 9), колонки: ['Sex', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight', 'Rings']
insurance: (1338, 7), колонки: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'expenses']


In [7]:
ctgan_space = {
    # Learning rates
    'discriminator_lr': hp.qloguniform('discriminator_lr',
                                                 np.log(4e-4), np.log(2.1e-3), 5e-5),
    'generator_lr': hp.qloguniform('generator_lr',
                                             np.log(5e-5), np.log(5e-3), 5e-5),

    # Batch size
    'batch_size': hp.choice('batch_size', [100, 500, 1000]),

    # Embedding dimensions
    'embedding_dim': hp.choice('embedding_dim', [32, 128]),

    # Network architecture
    'generator_dim': hp.choice('generator_dim', [[128, 128, 128], [128, 128, 128, 128]]),
    'discriminator_dim': hp.choice('discriminator_dim', [[256, 256], [256, 256, 256]]),

    # Decay parameters
    'generator_decay': hp.qloguniform('generator_decay',
                                     np.log(1e-6), np.log(6.4e-6), 1e-7),
    'discriminator_decay': hp.qloguniform('discriminator_decay',
                                         np.log(1e-6), np.log(8e-6), 1e-6),

    # Frequency
    'log_frequency': hp.choice('log_frequency', [False, True]),


    # Training epochs
    'epochs':hp.choice('epochs', [400])
}

In [8]:
def evaluate_c2st(real, synthetic):
    real_copy = real.copy()
    synthetic_copy = synthetic.copy()

    # Добавление меток
    real_copy['label'] = 1
    synthetic_copy['label'] = 0

    # Объединение данных
    df = pd.concat([real_copy, synthetic_copy], ignore_index=True)

    # Определение категориальных колонок
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
    categorical_columns = [col for col in categorical_columns if col != 'label']

    # Разделение на признаки и метки
    X = df.drop(columns='label')
    y = df['label']

    # Разделение на обучающую и тестовую выборки
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=RANDOM_SEED
    )

    # Обучение CatBoost
    clf = CatBoostClassifier(
        random_seed=RANDOM_SEED,
        verbose=0
    )

    clf.fit(X_train, y_train, cat_features=categorical_columns, eval_set=(X_test, y_test))

    # Предсказание и вычисление метрик
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    return f1 #accuracy


In [9]:
def generate_and_evaluate(params, dataset):
    #Извлечение метаданных
    metadata = SingleTableMetadata()
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    metadata.detect_from_dataframe(data=dataset)

    # Создание и обучение TVAE с заданными параметрами
    ctgan = CTGAN(
        metadata=metadata,
        discriminator_lr=params['discriminator_lr'],
        generator_lr=params['generator_lr'],
        batch_size=params['batch_size'],
        embedding_dim=params['embedding_dim'],
        generator_dim=params['generator_dim'],
        generator_decay=params['generator_decay'],
        discriminator_decay=params['discriminator_decay'],
        log_frequency=params['log_frequency'],
        #epochs=int(params['epochs'])
    )

    # Обучение модели на реальных данных
    ctgan.fit(dataset)

    # Генерация синтетических данных
    synthetic_data = ctgan.sample(len(dataset))

    # Вычисление метрики C2ST
    c2st_score = evaluate_c2st(dataset, synthetic_data)

    return c2st_score

In [10]:
def optimize_dataset(dataset_name, dataset, max_evals=30):
    print(f"Оптимизация для датасета: {dataset_name}, размер: {dataset.shape}")

    # Создаем объект для хранения истории поиска
    trials = Trials()

    # Определяем целевую функцию для оптимизации
    def objective(params):
        print("="*50)
        c2st_score = generate_and_evaluate(params, dataset)

        print(c2st_score)
        print(params)
        print("="*50)
        # Возвращаем словарь в формате, который ожидает hyperopt
        return {
            'loss': c2st_score,
            'status': STATUS_OK,
            'c2st_score': c2st_score
        }


    # Запускаем оптимизацию с помощью TPE алгоритма
    rng = np.random.default_rng(RANDOM_SEED)
    best = fmin(
        fn=objective,
        space=ctgan_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=rng
    )

    # Получаем лучшие параметры в читаемом виде
    best_params = {
        'discriminator_lr': best['discriminator_lr'],
        'generator_lr': best['generator_lr'],
        'batch_size': [100, 500, 1000][best['batch_size']],
        'embedding_dim': [32, 128][best['embedding_dim']],
        'generator_dim': [[128, 128, 128], [128, 128, 128, 128]][best['generator_dim']],
        'discriminator_dim': [[256, 256], [256, 256, 256]][best['discriminator_dim']],
        'generator_decay': best['generator_decay'],
        'discriminator_decay': best['discriminator_decay'],
        'log_frequency': [False, True][best['log_frequency']],
        'epochs': float('inf')
    }

    # Находим лучший результат
    best_trial = min(trials.trials, key=lambda x: x['result']['loss'])
    best_c2st = best_trial['result'].get('c2st_score', None)

    results = {
        'best_params': best_params,
        'best_c2st': best_c2st,
        'c2st_diff_from_optimal': best_trial['result']['loss']
    }

    print(f"Лучший C2ST для {dataset_name}: {best_c2st}")
    print(f"Лучшие параметры: {best_params}")
    print('-' * 50)

    return results

In [11]:
# Словарь для хранения результатов
all_results = {}

# Указываем количество итераций для каждого датасета
max_evals = 50

# Запускаем оптимизацию для каждого датасета
for dataset_name in datasets:
    data = datasets[dataset_name]
    all_results[dataset_name] = optimize_dataset(dataset_name, data, max_evals)

Оптимизация для датасета: abolone, размер: (4177, 9)
0.9952267303102625
{'batch_size': 100, 'discriminator_decay': 1e-06, 'discriminator_dim': (256, 256, 256), 'discriminator_lr': 0.0014500000000000001, 'embedding_dim': 32, 'epochs': 400, 'generator_decay': 5.199999999999999e-06, 'generator_dim': (128, 128, 128, 128), 'generator_lr': 0.0008500000000000001, 'log_frequency': False}
0.9936406995230525
{'batch_size': 1000, 'discriminator_decay': 1e-06, 'discriminator_dim': (256, 256), 'discriminator_lr': 0.001, 'embedding_dim': 128, 'epochs': 400, 'generator_decay': 4.9e-06, 'generator_dim': (128, 128, 128), 'generator_lr': 0.0008, 'log_frequency': True}
0.98698224852071
{'batch_size': 1000, 'discriminator_decay': 6e-06, 'discriminator_dim': (256, 256), 'discriminator_lr': 0.0010500000000000002, 'embedding_dim': 128, 'epochs': 400, 'generator_decay': 3.1e-06, 'generator_dim': (128, 128, 128), 'generator_lr': 5e-05, 'log_frequency': False}
0.9920571882446386
{'batch_size': 1000, 'discrimina